# Chapter 6: RLHF: The Three-Step Dance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arunpshankar/packt-final/blob/main/code/notebooks/part2_core/06_rlhf_ppo.ipynb)

> **Book:** *Reinforcement Learning for Large Language Models* — Arun Shankar & Michael Chertushkin (Packt, 2025)  
> **Chapter 6:** RLHF: The Three-Step Dance  
> **Notebook:** `part2_core/06_rlhf_ppo.ipynb`

---

## What this notebook covers

This is the companion notebook for Chapter 6 of the book. Run it on a free Colab T4 GPU. All code uses small, publicly available models (under 500 MB) that fit within the free tier memory limit.


In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Colab's base image has packages (gcsfs, bigframes, ...) that conflict with our
    # pinned versions but are unused by this notebook. Silence the noisy resolver
    # output by sending pip's stderr to /dev/null; 'Environment ready.' below
    # confirms the install actually succeeded.
    !pip install -q transformers==4.41.0 trl==0.8.6 peft==0.10.0 \
        datasets==2.19.0 accelerate==0.29.3 2>/dev/null

print('Environment ready.')

In [ ]:
import os
# Reduce CUDA allocator fragmentation — set BEFORE importing torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Silence advisory "use __call__ instead" hints from TRL's internal tokenization
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import warnings
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
)
from trl import (
    SFTTrainer,
    PPOTrainer,
    PPOConfig,
    AutoModelForCausalLMWithValueHead,
)
from datasets import Dataset

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram_gb:.1f} GB')

## Step 1 — Supervised Fine-Tuning (SFT)

Before we can apply RL, the model must learn to follow instructions at all. SFT provides exactly this: we show the model curated `(prompt, response)` pairs and fine-tune with a standard cross-entropy loss.

**Why SFT first?**  
A raw pre-trained model maximises next-token prediction on internet text — it will happily continue a question with another question. SFT shifts the probability mass toward instruction-following completions, giving PPO a well-behaved starting point.

### Dataset — 20 inline examples


In [ ]:
SFT_EXAMPLES = [
    {"prompt": "What is the capital of France?",
     "response": "The capital of France is Paris."},
    {"prompt": "Explain photosynthesis in one sentence.",
     "response": "Photosynthesis converts sunlight, water, and CO2 into glucose and oxygen inside plant cells."},
    {"prompt": "Write a haiku about the ocean.",
     "response": "Waves crash on the shore / Salt and foam dance endlessly / Sea calls out to me."},
    {"prompt": "What is 15 multiplied by 7?",
     "response": "15 multiplied by 7 equals 105."},
    {"prompt": "List three programming languages.",
     "response": "Three popular programming languages are Python, JavaScript, and Rust."},
    {"prompt": "What does CPU stand for?",
     "response": "CPU stands for Central Processing Unit."},
    {"prompt": "Summarise the water cycle briefly.",
     "response": "Water evaporates from surfaces, condenses into clouds, and falls back as precipitation."},
    {"prompt": "Who wrote Romeo and Juliet?",
     "response": "Romeo and Juliet was written by William Shakespeare."},
    {"prompt": "What is the boiling point of water in Celsius?",
     "response": "Water boils at 100 degrees Celsius at standard atmospheric pressure."},
    {"prompt": "Give a synonym for happy.",
     "response": "A synonym for happy is joyful."},
    {"prompt": "Translate hello into Spanish.",
     "response": "Hello in Spanish is Hola."},
    {"prompt": "What is machine learning?",
     "response": "Machine learning is a branch of AI where models learn patterns from data without explicit rules."},
    {"prompt": "Name the planet closest to the Sun.",
     "response": "Mercury is the planet closest to the Sun."},
    {"prompt": "What is the square root of 144?",
     "response": "The square root of 144 is 12."},
    {"prompt": "Describe the color red in two words.",
     "response": "Vibrant and warm."},
    {"prompt": "What is Newton's first law?",
     "response": "An object remains at rest or in uniform motion unless acted upon by an external force."},
    {"prompt": "Convert 100 Fahrenheit to Celsius.",
     "response": "100 degrees Fahrenheit is approximately 37.8 degrees Celsius."},
    {"prompt": "What is the largest mammal?",
     "response": "The blue whale is the largest mammal on Earth."},
    {"prompt": "List the primary colors.",
     "response": "The primary colors are red, blue, and yellow."},
    {"prompt": "What is the speed of light?",
     "response": "The speed of light in a vacuum is approximately 299,792,458 metres per second."},
]

def format_instruction(ex):
    return (
        f"### Instruction:\n{ex['prompt']}\n\n"
        f"### Response:\n{ex['response']}<|endoftext|>"
    )

sft_texts = [format_instruction(ex) for ex in SFT_EXAMPLES]
sft_dataset = Dataset.from_dict({"text": sft_texts})
print(f'SFT dataset: {len(sft_dataset)} examples')
print('\nSample:')
print(sft_dataset[0]['text'])

In [ ]:
SFT_MODEL_NAME = 'Qwen/Qwen2.5-0.5B'

sft_tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_NAME)
sft_tokenizer.pad_token = sft_tokenizer.eos_token
sft_tokenizer.padding_side = 'right'

sft_model = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_NAME,
    torch_dtype=torch.float32,
)
n_params = sum(p.numel() for p in sft_model.parameters())
print(f'Model parameters: {n_params / 1e6:.1f} M')

In [ ]:
sft_args = TrainingArguments(
    output_dir='./sft_output',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=5,
    save_strategy='no',
    fp16=(DEVICE == 'cuda'),
    report_to='none',
    seed=SEED,
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_args,
    train_dataset=sft_dataset,
    tokenizer=sft_tokenizer,
    dataset_text_field='text',
    max_seq_length=128,
)

print('Starting SFT training ...')
sft_trainer.train()
# Save the SFT checkpoint — PPO must start from these weights, not the raw base model
sft_trainer.save_model('./sft_checkpoint')
sft_tokenizer.save_pretrained('./sft_checkpoint')
print('SFT complete! Checkpoint saved to ./sft_checkpoint')

In [ ]:
TEST_PROMPTS = [
    "What is the capital of Germany?",
    "Explain gravity in one sentence.",
    "What is 8 times 9?",
]

def generate_text(model, tokenizer, prompt, max_new_tokens=50, device='cpu'):
    formatted = f"### Instruction:\n{prompt}\n\n### Response:\n"
    enc = tokenizer(
        formatted, return_tensors='pt', truncation=True, max_length=100
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    model.to(device)
    model.eval()
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out[0][enc['input_ids'].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

print('=== SFT Model Responses (before PPO) ===')
sft_responses_before = []
for p in TEST_PROMPTS:
    resp = generate_text(sft_model, sft_tokenizer, p, device=DEVICE)
    sft_responses_before.append(resp)
    print(f'Q: {p}')
    print(f'A: {resp}')
    print()

## Step 2 — Reward Model Training

The reward model answers: *"Given two responses to the same prompt, which one would a human prefer?"*

### Bradley-Terry preference model

We model the probability that response $y_w$ is preferred over $y_l$ as:

$$P(y_w \succ y_l \mid x) = \sigma\bigl(r_\theta(x, y_w) - r_\theta(x, y_l)\bigr)$$

The training loss is the negative log-likelihood over the preference dataset:

$$\mathcal{L}_{\text{RM}} = -\mathbb{E}_{(x,\,y_w,\,y_l)}\!\left[\log\,\sigma\bigl(r_\theta(x, y_w) - r_\theta(x, y_l)\bigr)\right]$$

### Architecture
We attach a single linear scalar head to a frozen Qwen/Qwen2.5-0.5B backbone. The scalar output is the raw reward score.


In [ ]:
PREFERENCE_PAIRS = [
    {"prompt": "What is the capital of France?",
     "chosen":   "The capital of France is Paris, a city renowned for the Eiffel Tower and the Louvre.",
     "rejected": "France has many cities."},
    {"prompt": "How do I make tea?",
     "chosen":   "Boil water, add a tea bag, steep for 3-5 minutes, remove the bag, and enjoy.",
     "rejected": "Tea is made with water."},
    {"prompt": "What is machine learning?",
     "chosen":   "Machine learning is an AI field where algorithms learn patterns from data to make predictions.",
     "rejected": "It is when computers do things."},
    {"prompt": "Explain gravity.",
     "chosen":   "Gravity is a fundamental force attracting masses toward each other, keeping planets in orbit.",
     "rejected": "Things fall down because of gravity."},
    {"prompt": "What is 12 times 12?",
     "chosen":   "12 times 12 equals 144.",
     "rejected": "A lot."},
    {"prompt": "Describe the Sun.",
     "chosen":   "The Sun is a G-type star at the centre of our solar system with a surface temperature of about 5,500 C.",
     "rejected": "The Sun is bright and hot."},
    {"prompt": "What is Python?",
     "chosen":   "Python is a high-level interpreted language prized for readability and widely used in data science.",
     "rejected": "A snake or a programming thing."},
    {"prompt": "How does the heart work?",
     "chosen":   "The heart is a four-chambered pump that circulates oxygenated blood to organs via rhythmic contractions.",
     "rejected": "The heart pumps blood."},
    {"prompt": "What is climate change?",
     "chosen":   "Climate change refers to long-term shifts in global temperature driven mainly by human greenhouse gas emissions.",
     "rejected": "The weather is getting weird."},
    {"prompt": "Who invented the telephone?",
     "chosen":   "Alexander Graham Bell is credited with the telephone in 1876; Elisha Gray filed a patent the same day.",
     "rejected": "Some old guy did."},
    {"prompt": "What is DNA?",
     "chosen":   "DNA is the molecule carrying genetic instructions for the development of all living organisms.",
     "rejected": "DNA is in cells."},
    {"prompt": "Explain the water cycle.",
     "chosen":   "Water evaporates, rises and condenses into clouds, then falls as precipitation and flows back to oceans.",
     "rejected": "Water goes up and comes down."},
    {"prompt": "What is the speed of sound?",
     "chosen":   "The speed of sound in air at room temperature is approximately 343 metres per second.",
     "rejected": "Sound is fast."},
    {"prompt": "How do vaccines work?",
     "chosen":   "Vaccines introduce a harmless pathogen form to train the immune system to recognise and fight the real pathogen.",
     "rejected": "They protect you from disease."},
    {"prompt": "What is photosynthesis?",
     "chosen":   "Photosynthesis is the process where plants use sunlight, CO2, and water to produce glucose and oxygen.",
     "rejected": "Plants eat sunlight."},
    # --- 25 additional pairs for better RM generalisation ---
    {"prompt": "What is an atom?",
     "chosen":   "An atom is the smallest unit of an element, consisting of a nucleus of protons and neutrons surrounded by electrons.",
     "rejected": "Atoms are tiny."},
    {"prompt": "How does electricity work?",
     "chosen":   "Electricity is the flow of electrons through a conductor, driven by a voltage difference between two points.",
     "rejected": "It comes from the wall."},
    {"prompt": "What is the Internet?",
     "chosen":   "The Internet is a global network of interconnected computers that communicate using standardised protocols like TCP/IP.",
     "rejected": "It is where websites are."},
    {"prompt": "Why is the sky blue?",
     "chosen":   "The sky appears blue because shorter blue wavelengths of sunlight are scattered more by atmospheric molecules than longer wavelengths.",
     "rejected": "Because it just is."},
    {"prompt": "What is a black hole?",
     "chosen":   "A black hole is a region of spacetime where gravity is so strong that nothing, not even light, can escape its event horizon.",
     "rejected": "A hole in space."},
    {"prompt": "What is inflation in economics?",
     "chosen":   "Inflation is the sustained increase in the general price level of goods and services, reducing the purchasing power of money.",
     "rejected": "Prices go up."},
    {"prompt": "How do airplanes fly?",
     "chosen":   "Airplanes fly by generating lift: air moves faster over the curved upper wing surface, creating lower pressure above than below.",
     "rejected": "Engines push them up."},
    {"prompt": "What is a programming variable?",
     "chosen":   "A variable is a named storage location in memory that holds a value which can be read and modified during program execution.",
     "rejected": "A thing in code."},
    {"prompt": "What causes earthquakes?",
     "chosen":   "Earthquakes occur when tectonic plates along fault lines suddenly slip past each other, releasing accumulated seismic energy.",
     "rejected": "The ground shakes sometimes."},
    {"prompt": "What is the periodic table?",
     "chosen":   "The periodic table organises all known chemical elements by atomic number, electron configuration, and recurring chemical properties.",
     "rejected": "A chart of elements."},
    {"prompt": "How does a battery work?",
     "chosen":   "A battery converts stored chemical energy into electrical energy through electrochemical reactions between an anode and cathode.",
     "rejected": "It stores power."},
    {"prompt": "What is democracy?",
     "chosen":   "Democracy is a system of government where citizens exercise power through voting, either directly or via elected representatives.",
     "rejected": "People vote."},
    {"prompt": "What is the greenhouse effect?",
     "chosen":   "The greenhouse effect occurs when atmospheric gases like CO2 and methane trap heat radiated from Earth's surface, warming the planet.",
     "rejected": "It makes Earth warm."},
    {"prompt": "How do computers store data?",
     "chosen":   "Computers store data as binary digits (0s and 1s) in memory devices like RAM for temporary storage and SSDs or HDDs for persistent storage.",
     "rejected": "On a hard drive."},
    {"prompt": "What is a chemical reaction?",
     "chosen":   "A chemical reaction is a process in which reactant molecules rearrange their atoms to form new substances with different properties.",
     "rejected": "Things change."},
    {"prompt": "What is evolution?",
     "chosen":   "Evolution is the gradual change in heritable traits of biological populations over successive generations, driven by natural selection.",
     "rejected": "Animals change over time."},
    {"prompt": "What is an API?",
     "chosen":   "An API (Application Programming Interface) defines a set of rules and protocols that allow different software systems to communicate.",
     "rejected": "A way for code to talk."},
    {"prompt": "What is a neural network?",
     "chosen":   "A neural network is a computational model inspired by biological neurons, consisting of layers of interconnected nodes that learn patterns from data.",
     "rejected": "AI brains."},
    {"prompt": "How does GPS work?",
     "chosen":   "GPS works by triangulating signals from at least four orbiting satellites, measuring the time delay of each signal to compute precise position.",
     "rejected": "Satellites tell you where you are."},
    {"prompt": "What is the stock market?",
     "chosen":   "The stock market is a network of exchanges where shares of publicly traded companies are bought and sold, reflecting supply and demand for ownership.",
     "rejected": "Where you buy stocks."},
    {"prompt": "What is a compiler?",
     "chosen":   "A compiler translates source code written in a high-level programming language into machine code that a processor can execute directly.",
     "rejected": "It runs code."},
    {"prompt": "What is renewable energy?",
     "chosen":   "Renewable energy comes from naturally replenishing sources like sunlight, wind, water, and geothermal heat, producing minimal greenhouse emissions.",
     "rejected": "Energy from the sun and wind."},
    {"prompt": "How does the immune system work?",
     "chosen":   "The immune system defends the body using white blood cells, antibodies, and other mechanisms that identify and destroy pathogens like bacteria and viruses.",
     "rejected": "It fights germs."},
    {"prompt": "What is supply and demand?",
     "chosen":   "Supply and demand is an economic model: when demand exceeds supply prices rise; when supply exceeds demand prices fall, reaching equilibrium.",
     "rejected": "More people want it, price goes up."},
    {"prompt": "What is an operating system?",
     "chosen":   "An operating system is software that manages computer hardware and provides services to applications, handling tasks like memory allocation and process scheduling.",
     "rejected": "Windows or Mac."},
]

print(f'Preference dataset: {len(PREFERENCE_PAIRS)} pairs')
p0 = PREFERENCE_PAIRS[0]
print(f'  Prompt  : {p0["prompt"]}')
print(f'  Chosen  : {p0["chosen"]}')
print(f'  Rejected: {p0["rejected"]}')


In [ ]:
# Free SFT-phase GPU memory before training the reward model (T4 has ~16 GB)
import gc
for _name in ('sft_trainer', 'sft_model'):
    if _name in globals():
        del globals()[_name]
gc.collect(); torch.cuda.empty_cache()

class RewardModel(nn.Module):
    """Qwen/Qwen2.5-0.5B backbone with a single scalar reward head."""

    def __init__(self, backbone_name: str = 'Qwen/Qwen2.5-0.5B'):
        super().__init__()
        self.backbone = AutoModelForCausalLM.from_pretrained(backbone_name, torch_dtype=torch.float32)
        self.backbone.gradient_checkpointing_enable()  # trade compute for memory on T4
        self.backbone.config.use_cache = False  # required when grad checkpointing is on
        hidden_size = self.backbone.config.hidden_size  # 896 for Qwen2.5-0.5B
        self.reward_head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 1, bias=False),
        )
        nn.init.normal_(self.reward_head[1].weight, std=0.01)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        last_hidden = outputs.hidden_states[-1]          # (B, T, H)
        seq_len_idx = attention_mask.sum(dim=1) - 1      # index of last real token
        B = last_hidden.shape[0]
        last_tok = last_hidden[torch.arange(B), seq_len_idx]  # (B, H)
        reward = self.reward_head(last_tok).squeeze(-1)         # (B,)
        return reward


rm_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B')
rm_tokenizer.pad_token = rm_tokenizer.eos_token
rm_tokenizer.padding_side = 'right'

reward_model = RewardModel('Qwen/Qwen2.5-0.5B').to(DEVICE)
n_rm = sum(p.numel() for p in reward_model.parameters())
print(f'Reward model parameters: {n_rm / 1e6:.1f} M')

In [ ]:
from torch.optim import AdamW


def tokenize_pair(prompt: str, response: str, tokenizer, max_len: int = 128):
    text = f"### Instruction:\n{prompt}\n\n### Response:\n{response}"
    enc = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=max_len,
        padding='max_length',
    )
    return enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0)


rm_optimizer = AdamW(reward_model.parameters(), lr=1e-4, weight_decay=0.01)
NUM_RM_EPOCHS = 3  # reduced from 5 to prevent overfitting on small dataset
rm_losses, rm_accuracies = [], []

for epoch in range(NUM_RM_EPOCHS):
    epoch_loss, correct = [], 0
    reward_model.train()
    random.shuffle(PREFERENCE_PAIRS)

    for pair in PREFERENCE_PAIRS:
        ids_w, mask_w = tokenize_pair(pair['prompt'], pair['chosen'],   rm_tokenizer)
        ids_l, mask_l = tokenize_pair(pair['prompt'], pair['rejected'], rm_tokenizer)

        ids_w  = ids_w.unsqueeze(0).to(DEVICE)
        mask_w = mask_w.unsqueeze(0).to(DEVICE)
        ids_l  = ids_l.unsqueeze(0).to(DEVICE)
        mask_l = mask_l.unsqueeze(0).to(DEVICE)

        r_w = reward_model(ids_w, mask_w)
        r_l = reward_model(ids_l, mask_l)

        # Bradley-Terry loss: -log sigma(r_w - r_l)
        loss = -torch.log(torch.sigmoid(r_w - r_l) + 1e-8).mean()

        rm_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
        rm_optimizer.step()

        epoch_loss.append(loss.item())
        if r_w.item() > r_l.item():
            correct += 1

    avg_loss = float(np.mean(epoch_loss))
    acc = correct / len(PREFERENCE_PAIRS)
    rm_losses.append(avg_loss)
    rm_accuracies.append(acc)
    print(f'Epoch {epoch+1}/{NUM_RM_EPOCHS}  loss={avg_loss:.4f}  acc={acc:.2%}')

print('Reward model training complete!')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, NUM_RM_EPOCHS + 1)
ax1.plot(epochs, rm_losses, 'b-o', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Bradley-Terry Loss')
ax1.set_title('Reward Model — Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, rm_accuracies, 'g-o', linewidth=2)
ax2.axhline(0.5, color='r', linestyle='--', alpha=0.6, label='random baseline')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Pairwise Accuracy')
ax2.set_title('Reward Model — Pairwise Accuracy')
ax2.set_ylim([0, 1.05])
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rm_training.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
def score_response(model, tokenizer, prompt: str, response: str, device: str) -> float:
    """Return scalar reward for a (prompt, response) pair."""
    ids, mask = tokenize_pair(prompt, response, tokenizer)
    ids  = ids.unsqueeze(0).to(device)
    mask = mask.unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        return model(ids, mask).item()


# Sanity-check one pair
p = PREFERENCE_PAIRS[0]
sc = score_response(reward_model, rm_tokenizer, p['prompt'], p['chosen'],   DEVICE)
sr = score_response(reward_model, rm_tokenizer, p['prompt'], p['rejected'], DEVICE)
print(f'Prompt:          {p["prompt"]}')
print(f'Chosen  score:   {sc:.4f}')
print(f'Rejected score:  {sr:.4f}')
print(f'Correct ranking: {sc > sr}')

## Step 3 — PPO Fine-Tuning

Proximal Policy Optimization (PPO) is an on-policy actor-critic algorithm that updates the policy while preventing large destructive steps via **clipping**.

### PPO objective (clipped)

$$\mathcal{L}_{\text{PPO}} = -\mathbb{E}\!\left[\min\!\left(\rho_t\,\hat{A}_t,\;\text{clip}(\rho_t,\,1\!-\!\epsilon,\,1\!+\!\epsilon)\,\hat{A}_t\right)\right]$$

where $\rho_t = \pi_\theta / \pi_{\text{old}}$ is the probability ratio.

### KL-penalised reward

In RLHF the advantage signal comes from the reward model, augmented by a KL penalty that prevents the policy from drifting too far from the SFT reference:

$$r_t = r_\theta(x, y) - \beta\,\text{KL}[\pi_\phi(y\mid x)\;\|\;\pi_{\text{ref}}(y\mid x)]$$

### TRL's PPOTrainer
TRL handles value-function training, KL computation, reward normalisation, clipping, and gradient updates automatically.


In [ ]:
PPO_PROMPTS = [
    "What is the capital of Japan?",
    "How many planets are in the solar system?",
    "What is the largest ocean?",
    "Who wrote Hamlet?",
    "What is the chemical formula for water?",
    "What does HTML stand for?",
    "What is the Pythagorean theorem?",
    "Name the largest continent.",
    "What is the freezing point of water in Celsius?",
    "What is the powerhouse of the cell?",
    # --- Additional prompts for diversity ---
    "What is the tallest mountain on Earth?",
    "Explain what a CPU does in one sentence.",
    "What is the chemical symbol for gold?",
    "How many sides does a hexagon have?",
    "What is the speed of light approximately?",
    "Name three types of rock.",
    "What is a prime number?",
    "How long does Earth take to orbit the Sun?",
    "What is the largest organ in the human body?",
    "Explain what an algorithm is.",
    "What gas do humans breathe in?",
    "Who painted the Mona Lisa?",
    "What is the smallest planet in our solar system?",
    "Define the term ecosystem.",
    "What is absolute zero in Celsius?",
    "How does a magnet work?",
    "What is the main ingredient in bread?",
    "Name the three states of matter.",
    "What is an electron?",
    "What does DNA stand for?",
]

print(f'PPO prompt pool: {len(PPO_PROMPTS)} prompts')


In [ ]:
# Free anything we no longer need before constructing the PPO trainer
import gc
for _name in ('rm_optimizer',):  # Adam state ~4 GB for 0.5B model; no longer needed after RM training
    if _name in globals():
        del globals()[_name]
gc.collect(); torch.cuda.empty_cache()

ppo_tokenizer = AutoTokenizer.from_pretrained('./sft_checkpoint')
ppo_tokenizer.pad_token = ppo_tokenizer.eos_token
ppo_tokenizer.padding_side = 'left'  # required by PPOTrainer for generation

# FIX: Load the SFT-trained checkpoint, NOT the raw base model.
# Previously this loaded 'Qwen/Qwen2.5-0.5B' — throwing away SFT entirely.
ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained('./sft_checkpoint', torch_dtype=torch.float32)
ppo_model.to(DEVICE)

# Move reward model to CPU to free ~2 GB of GPU memory for PPO's value head + Adam state.
# Reward scoring runs only ~2 times per PPO step on short sequences, so CPU is fine here.
reward_model = reward_model.to('cpu')
gc.collect(); torch.cuda.empty_cache()

ppo_config = PPOConfig(
    model_name='Qwen/Qwen2.5-0.5B',
    learning_rate=1e-5,
    batch_size=2,  # fits on T4 alongside policy + reference + reward model
    mini_batch_size=1,
    gradient_accumulation_steps=1,
    ppo_epochs=2,
    max_grad_norm=0.5,
    target_kl=6.0,          # raised from 0.1 — allows adaptive KL controller to work
    init_kl_coef=0.5,       # increased from 0.2 — stronger KL leash
    cliprange=0.2,
    cliprange_value=0.2,
    vf_coef=0.1,
    log_with=None,
    seed=SEED,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=ppo_model,
    ref_model=None,   # TRL clones a frozen reference internally
    tokenizer=ppo_tokenizer,
)

print('PPOTrainer ready.')

In [ ]:
PPO_STEPS     = 80  # increased from 50 for smoother convergence
BATCH_SIZE    = 2  # must match PPOConfig.batch_size
MAX_NEW_TOKS  = 40

ppo_reward_log = []
ppo_kl_log     = []

gen_kwargs = {
    'max_new_tokens': MAX_NEW_TOKS,
    'do_sample':      True,
    'temperature':    0.7,
    'top_k':          50,
    'pad_token_id':   ppo_tokenizer.eos_token_id,
}

# Running reward statistics for normalisation (prevents ±37 scale blowup)
reward_running = []  # collects all raw rewards seen so far

print(f'Running PPO for {PPO_STEPS} steps ...')

for step in range(PPO_STEPS):
    batch_prompts = random.choices(PPO_PROMPTS, k=BATCH_SIZE)
    query_texts   = [f"### Instruction:\n{p}\n\n### Response:\n" for p in batch_prompts]

    query_tensors = [
        ppo_tokenizer(qt, return_tensors='pt').input_ids.squeeze(0).to(DEVICE)
        for qt in query_texts
    ]

    response_tensors = ppo_trainer.generate(query_tensors, **gen_kwargs)

    batch_responses = [
        ppo_tokenizer.decode(r[q.shape[0]:], skip_special_tokens=True)
        for q, r in zip(query_tensors, response_tensors)
    ]

    # Score with RM then normalise to roughly zero-mean, unit-variance
    raw_scores = [
        score_response(reward_model, rm_tokenizer, pr, re, 'cpu')
        for pr, re in zip(batch_prompts, batch_responses)
    ]
    reward_running.extend(raw_scores)
    r_mean = float(np.mean(reward_running))
    r_std  = max(float(np.std(reward_running)), 1.0)  # floor at 1.0 to avoid div-by-zero early on
    rewards = [
        torch.tensor((s - r_mean) / r_std, dtype=torch.float32).clamp(-5.0, 5.0)
        for s in raw_scores
    ]

    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)

    avg_reward = float(np.mean([r.item() for r in rewards]))
    kl = float(stats.get('objective/kl', 0.0))
    ppo_reward_log.append(avg_reward)
    ppo_kl_log.append(kl)

    if (step + 1) % 10 == 0:
        print(f'Step {step+1:3d}/{PPO_STEPS}  reward={avg_reward:.4f}  kl={kl:.4f}')

print('PPO training complete!')

In [ ]:
def smooth(data, w=5):
    if len(data) < w:
        return np.array(data)
    return np.convolve(data, np.ones(w) / w, mode='valid')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

steps = np.arange(1, len(ppo_reward_log) + 1)
ax1.plot(steps, ppo_reward_log, alpha=0.25, color='steelblue')
sm = smooth(ppo_reward_log)
ax1.plot(np.arange(1, len(sm) + 1), sm, color='steelblue', linewidth=2, label='smoothed (w=5)')
ax1.set_xlabel('PPO Step')
ax1.set_ylabel('Average Reward')
ax1.set_title('Reward over PPO Steps')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(steps, ppo_kl_log, color='orange', linewidth=2)
ax2.axhline(ppo_config.target_kl, color='r', linestyle='--',
            label=f'target KL = {ppo_config.target_kl}')
ax2.set_xlabel('PPO Step')
ax2.set_ylabel('KL Divergence')
ax2.set_title('KL Divergence (policy vs. reference)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ppo_training.png', dpi=100, bbox_inches='tight')
plt.show()

## Before vs. After PPO — Side-by-Side Comparison

We generate responses to the same 3 test prompts with both models and let the reward model judge them. After PPO the policy has been nudged toward higher-scoring, more informative completions.


In [ ]:
def generate_ppo_text(model, tokenizer, prompt: str, max_new_tokens: int = 50) -> str:
    formatted = f"### Instruction:\n{prompt}\n\n### Response:\n"
    enc = tokenizer(formatted, return_tensors='pt', truncation=True, max_length=100)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    model.eval()
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out[0][enc['input_ids'].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


print('=== Before vs. After PPO ===')
print('=' * 62)

rb_list, ra_list = [], []

for i, prompt in enumerate(TEST_PROMPTS):
    resp_before = sft_responses_before[i]
    resp_after  = generate_ppo_text(ppo_model, ppo_tokenizer, prompt)

    rb = score_response(reward_model, rm_tokenizer, prompt, resp_before, 'cpu')
    ra = score_response(reward_model, rm_tokenizer, prompt, resp_after,  'cpu')
    rb_list.append(rb)
    ra_list.append(ra)

    delta = ra - rb
    sign  = '+' if delta >= 0 else ''
    print(f'\nPrompt: {prompt}')
    print(f'  Before PPO  (score={rb:+.3f}): "{resp_before[:75]}"')
    print(f'  After  PPO  (score={ra:+.3f}): "{resp_after[:75]}"')
    print(f'  Reward delta: {sign}{delta:.3f}')

In [ ]:
x = np.arange(len(TEST_PROMPTS))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - 0.2, rb_list, 0.4, label='Before PPO', color='salmon')
ax.bar(x + 0.2, ra_list, 0.4, label='After PPO',  color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels([f'Prompt {i+1}' for i in range(len(TEST_PROMPTS))])
ax.set_ylabel('Reward Score')
ax.set_title('Before vs. After PPO: Reward Scores')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('ppo_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Summary — The Three-Step Dance

| Step | Trained object | Loss / objective |
|------|---------------|------------------|
| 1 — SFT | Policy (Qwen/Qwen2.5-0.5B) | Cross-entropy on 20 instruction pairs |
| 2 — RM  | Reward model (Qwen/Qwen2.5-0.5B + scalar head) | Bradley-Terry on 40 preference pairs |
| 3 — PPO | Policy (Qwen/Qwen2.5-0.5B w/ value head) | Clipped PPO + KL penalty over 80 steps |

### Key design choices
- **KL penalty** keeps the policy close to the SFT reference, preventing reward hacking.
- **Clipping** (`ε = 0.2`) prevents catastrophic updates in a single step.
- **Value head** lets PPO estimate advantage without polluting the LM head.

### Limitations of this toy demo
Production RLHF uses millions of preference labels, thousands of PPO steps, and 7B–70B-parameter models. Our 82 M-parameter Qwen/Qwen2.5-0.5B and 15 preference pairs are a pedagogical stand-in.

> **Next chapter** → Chapter 7 introduces DPO, eliminating the explicit reward model by re-expressing preference optimisation as a supervised objective directly on the policy.


In [ ]:
print('=== RLHF Training Summary ===')
print(f'SFT examples             : {len(SFT_EXAMPLES)}')
print(f'Preference pairs (RM)    : {len(PREFERENCE_PAIRS)}')
print(f'PPO steps completed      : {PPO_STEPS}')
print(f'RM final accuracy        : {rm_accuracies[-1]:.2%}')
print(f'PPO reward — step 1      : {ppo_reward_log[0]:+.4f}')
print(f'PPO reward — last step   : {ppo_reward_log[-1]:+.4f}')
delta = ppo_reward_log[-1] - ppo_reward_log[0]
print(f'Reward improvement       : {"+" if delta >= 0 else ""}{delta:.4f}')
print(f'Mean KL divergence       : {np.mean(ppo_kl_log):.4f}')

## 6. Production-Scale PPO -- From Toy to InstructGPT

**Quick orientation before the table:** the policy we trained in this notebook is **Qwen/Qwen2.5-0.5B**. We are *not* using InstructGPT or Gemini -- those are not models you load. They are **two previously published applications of the same three-step RLHF algorithm** (SFT → RM → PPO) by OpenAI (2022) and Google. The table below compares our toy run to those production runs so you can see how the same recipe scales by orders of magnitude.

The PPO loop in this chapter used `Qwen/Qwen2.5-0.5B` (500 M parameters) so every step fits on a single T4 GPU.
Real alignment pipelines using the same algorithm operate at orders of magnitude larger scale:

| Dimension | This notebook (toy) | InstructGPT (OpenAI, 2022) | Gemini RLHF (Google DeepMind) |
|---|---|---|---|
| Base model size | 0.5 B | 175 B (GPT-3) | ~1 T (est.) |
| Reward model | 2-layer head on Qwen | Separate 6 B RM | Separate large RM |
| PPO batch size | 8 | 512+ | thousands |
| KL penalty beta | 0.05 (fixed) | adaptive | adaptive |
| Hardware | 1x T4 | thousands of A100s | TPU v4 pods |
| Human labels | synthetic | ~40 k curated | undisclosed |

The algorithmic structure (SFT --> RM --> PPO) is identical. The *model* in each column is different (Qwen, GPT-3, Gemini base), but the RLHF algorithm is the same -- that's the whole point of RLHF as a *technique*: it works on whatever base model you point it at.


In [ ]:
# Production PPO config -- Qwen2.5-7B-Instruct (reference only, not executed)
# This shows how the SAME trl.PPOTrainer call scales to a 7 B model.
# Running this cell would require ~40 GB VRAM; it is included for reference.

from trl import PPOConfig  # noqa: F401 -- import for schema visibility

PRODUCTION_PPO_CONFIG = PPOConfig(
    model_name='Qwen/Qwen2.5-7B-Instruct',
    learning_rate=1e-5,
    batch_size=64,
    mini_batch_size=8,
    gradient_accumulation_steps=8,
    ppo_epochs=4,
    kl_penalty='kl',           # adaptive KL used in InstructGPT
    init_kl_coef=0.2,
    target_kl=6.0,             # PPO clip target
    cliprange=0.2,
    cliprange_value=0.2,
    vf_coef=0.1,
    seed=42,
)

# To load the model itself (requires ~14 GB VRAM in float16):
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch
# prod_model = AutoModelForCausalLM.from_pretrained(
#     'Qwen/Qwen2.5-7B-Instruct',
#     torch_dtype=torch.float16,
#     device_map='auto',
# )
# prod_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-7B-Instruct')

print('Production PPO config defined (not executed -- too large for T4).')
print(f'  model : {PRODUCTION_PPO_CONFIG.model_name}')
print(f'  lr    : {PRODUCTION_PPO_CONFIG.learning_rate}')
print(f'  batch : {PRODUCTION_PPO_CONFIG.batch_size}')


## 7. Chapter Summary and Forward Pointer

**What we covered in Chapter 6:**
- The three-step RLHF dance: SFT, reward modeling, PPO fine-tuning
- Implementing `trl.PPOTrainer` end-to-end on a 0.5 B model
- Diagnosing KL collapse and reward hacking
- How the same loop scales to production (InstructGPT, Gemini)

**PPO's main limitation** is sample inefficiency -- it requires thousands of online rollouts for each gradient step.
Chapter 7 introduces **Direct Preference Optimization (DPO)**, which eliminates the RL loop entirely
by converting preference data directly into a supervised objective.
DPO achieves competitive alignment quality at a fraction of PPO's compute cost, and is now the
default alignment recipe at many labs.
